# Set Library Path

In [ ]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [ ]:
library(dplyr)
library(tidyr)
library(tidyverse)
library(enrichR)
library(openxlsx)
library(ggplot2)

# Gene Ontology

In [ ]:
# Get list of all cell lines from the module membership files
module_files <- list.files("08_WGCNA", pattern = "*_module_membership.txt", full.names = TRUE)
cell_lines <- gsub("08_WGCNA/|_module_membership.txt", "", module_files)

In [ ]:
# Process each cell line
for (cell_line in cell_lines) {
  cat(paste0("\nProcessing cell line: ", cell_line, "\n"))
  
  # Create cell line-specific directory
  cell_line_dir <- paste0("09_WGCNA_GO/", cell_line)
  if (!dir.exists(cell_line_dir)) {
    dir.create(cell_line_dir, recursive = TRUE)
  }
  
  # Read in modules for this cell line
  module_membership_file <- paste0("08_WGCNA/", cell_line, "_module_membership.txt")
  module_membership <- read.delim(module_membership_file, check.names = FALSE)
  
  # Save modules
  modules <- unique(module_membership$module_color)
  
  # Iterate over each module
  for (module in modules) {
    # Filter data for the current module
    module_data <- module_membership[module_membership$module_color == module, ]
    
    # Extract genes from gene name column
    genes <- module_data$gene_id[!is.na(module_data$gene_id) & module_data$gene_id != ""]
    
    # Create a data frame with a single column containing genes
    genes_df <- data.frame(Genes = genes)
    
    # Define the filename for the CSV
    filename <- paste0(cell_line_dir, "/", module, "_genes.csv")
    
    # Rename the column
    colnames(genes_df) <- module
    
    # Write the genes to a CSV file
    write.csv(genes_df, file = filename, row.names = FALSE)
  }
  
  # List to store genes for each module
  gene_lists <- list()
  
  # Iterate over each module
  for (module in modules) {
    # Define the filename for the CSV
    filename <- paste0(cell_line_dir, "/", module, "_genes.csv")
    
    # Read CSV file
    module_genes <- read.csv(filename, header = TRUE)[[1]] 
    
    # Store genes in the list
    gene_lists[[module]] <- module_genes
  }
  
  # Find the maximum length among all gene lists
  valid_lengths <- sapply(gene_lists, function(x) if (is.null(x) || length(x) == 0) 0 else length(x))
  max_length <- max(valid_lengths, na.rm = TRUE)
  
  # Pad shorter gene lists with NA values to make all lists equal length
  for (module in modules) {
    diff_length <- max_length - length(gene_lists[[module]])
    if (diff_length > 0) {
      gene_lists[[module]] <- c(gene_lists[[module]], rep(NA, diff_length))
    }
  }
  
  # Combine gene lists into a data frame
  merged_df <- as.data.frame(gene_lists)
  
  # Rename columns with module names
  colnames(merged_df) <- modules
  
  # Print the first few rows of the merged data frame
  head(merged_df)
  
  # Save all modules and genes into a dataframe
  write.csv(merged_df, file = paste0(cell_line_dir, "/", cell_line, "_all_genes_per_module.csv"), row.names = FALSE)
  
  # Iterate over each module
  for (module in modules) {
    tryCatch({
      # Perform the enrichR analysis on the gene list for the current module
      enrichr_results <- enrichr(merged_df[[module]], c("GO_Biological_Process_2025",
                                                       "GO_Cellular_Component_2025",
                                                       "GO_Molecular_Function_2025",
                                                       "KEGG_2021_Human",
                                                       "Panther_2016",
                                                       "Reactome_2022",
                                                       "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO"))
      
      # Check if enrichr_results is empty
      if (length(enrichr_results) == 0) {
        cat("No results for module", module, "in cell line", cell_line, "\n")
        next
      }
      
      # Save Enrichr outputs
      wb <- createWorkbook()
      
      for (i in seq_along(enrichr_results)) {
        # Extract the data frame from the list
        df <- enrichr_results[[i]]
        
        # Check if the data frame is empty
        if (nrow(df) == 0) {
          cat("Empty data frame for", names(enrichr_results)[i], "in module", module, "cell line", cell_line, "\n")
          next
        }
        
        # Define the original sheet name
        original_sheet_name <- names(enrichr_results)[i]
        
        # Modify the sheet name if it's specifically "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO"
        sheet_name <- if (original_sheet_name == "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO") {
          "RNAseq_DiseaseGene_DrugSigs_GEO"
        } else {
          original_sheet_name
        }
        
        # Add the data frame as a new sheet in the Excel workbook
        addWorksheet(wb, sheet_name)
        writeData(wb, sheet = sheet_name, x = df)
      }
      
      # Save the Excel workbook
      saveWorkbook(wb, paste0(cell_line_dir, "/", module, "_enrichr_results.xlsx"), overwrite = TRUE)
      
      # Function to plot and save the results
      plot_and_save <- function(df, filename, title) {
        if (nrow(df) == 0) {
          cat("Empty data frame for", title, "in module", module, "cell line", cell_line, "\n")
          return()
        }
        pdf(filename, height = 7, width = 15)
        print(plotEnrich(df, showTerms = 25, numChar = 75, y = "Count", orderBy = "P.value") + ggtitle(title))
        dev.off()
      }
      
      # Plot and save Enrichr results
      plot_and_save(enrichr_results$GO_Biological_Process_2025, 
                    paste0(cell_line_dir, "/", module, "_GO_Biological_Process_2025.pdf"), 
                    paste("GO_Biological_Process_2025 for", module, "module -", cell_line))
      
      plot_and_save(enrichr_results$GO_Cellular_Component_2025, 
                    paste0(cell_line_dir, "/", module, "_GO_Cellular_Component_2025.pdf"), 
                    paste("GO_Cellular_Component_2025 for", module, "module -", cell_line))
      
      plot_and_save(enrichr_results$GO_Molecular_Function_2025, 
                    paste0(cell_line_dir, "/", module, "_GO_Molecular_Function_2025.pdf"), 
                    paste("GO_Molecular_Function_2025 for", module, "module -", cell_line))
      
      plot_and_save(enrichr_results$KEGG_2021_Human, 
                    paste0(cell_line_dir, "/", module, "_KEGG_2021_Human.pdf"), 
                    paste("KEGG_2021_Human for", module, "module -", cell_line))
      
      plot_and_save(enrichr_results$Panther_2016, 
                    paste0(cell_line_dir, "/", module, "_Panther_2016.pdf"), 
                    paste("Panther_2016 for", module, "module -", cell_line))
      
      plot_and_save(enrichr_results$Reactome_2022, 
                    paste0(cell_line_dir, "/", module, "_Reactome_2022.pdf"), 
                    paste("Reactome_2022 for", module, "module -", cell_line))
      
      plot_and_save(enrichr_results$`RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO`, 
                    paste0(cell_line_dir, "/", module, "_RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO.pdf"), 
                    paste("RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO for", module, "module -", cell_line))
      
    }, error = function(e) {
      cat("Error occurred for module", module, "in cell line", cell_line, ": ", conditionMessage(e), "\n")
      # Log the error to a file for further inspection
      write(paste("Error occurred for module", module, "in cell line", cell_line, ": ", conditionMessage(e), "\n"), 
            file = paste0(cell_line_dir, "/error_log.txt"), append = TRUE)
      # Continue to the next module
      next
    })
  }
  
  cat(paste0("Completed processing for cell line: ", cell_line, "\n"))
}

cat("\nAll cell lines processed successfully!\n")